# DSA Week 7 -- Performance Sprint 1

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Focus:** Implement your optimized DSA feature

## Learning Objectives

1. Identify the performance bottleneck in your pipeline
2. Choose the right data structure to fix it
3. Implement an optimized version alongside the baseline
4. Write tests to prove correctness (optimized matches baseline)
5. Prepare for next week's benchmarking

## The Sprint Process

```
+-------------------+     +-------------------+     +-------------------+
|  1. IDENTIFY      | --> |  2. IMPLEMENT     | --> |  3. VERIFY        |
|  - Profile code   |     |  - Optimized ver  |     |  - Same results   |
|  - Find hotspot   |     |  - Keep baseline  |     |  - All tests pass |
|  - State Big-O    |     |  - Clean code     |     |  - Ready to bench |
+-------------------+     +-------------------+     +-------------------+
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Identify the Bottleneck

Before optimizing, you need to know WHAT is slow. Here is how to find bottlenecks:

### Method 1: Manual Timing
Wrap each pipeline stage with timing code:

In [ ]:
import time

def timed(func):
    """Decorator that prints execution time."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print("  " + func.__name__ + ": " + "{:.4f}".format(elapsed) + "s")
        return result
    return wrapper

# Example pipeline stages
@timed
def load_data(n):
    """Simulate loading n records."""
    return [{"id": i, "value": i * 1.5, "category": "cat_" + str(i % 10)} for i in range(n)]

@timed
def clean_data(data):
    """Remove invalid records."""
    return [r for r in data if r["value"] >= 0]

@timed
def find_duplicates(data):
    """Find duplicate values -- THIS IS THE BOTTLENECK."""
    seen = []  # BAD: using list for membership check
    dupes = []
    for r in data:
        val = r["value"]
        if val in seen:  # O(n) scan each time!
            dupes.append(r)
        seen.append(val)
    return dupes

@timed
def analyze(data):
    """Compute statistics."""
    values = [r["value"] for r in data]
    return {"count": len(values), "mean": sum(values) / len(values) if values else 0}

# Run pipeline
print("=== Pipeline Profiling (n=10,000) ===")
print()
data = load_data(10_000)
clean = clean_data(data)
dupes = find_duplicates(clean)
stats = analyze(clean)
print()
print("Bottleneck identified: find_duplicates is O(n^2) due to 'val in seen' on a list!")

---
## Part 2: Implement the Optimization

In [ ]:
@timed
def find_duplicates_optimized(data):
    """Find duplicate values using a set. O(n)."""
    seen = set()  # FIXED: O(1) membership check
    dupes = []
    for r in data:
        val = r["value"]
        if val in seen:  # O(1) check!
            dupes.append(r)
        seen.add(val)
    return dupes

# Verify correctness
print("=== Correctness Check ===")
print()
dupes_baseline = find_duplicates(clean)
dupes_optimized = find_duplicates_optimized(clean)

assert len(dupes_baseline) == len(dupes_optimized), "Must find same number of duplicates!"
print()
print("Both find " + str(len(dupes_baseline)) + " duplicates -- CORRECT!")
print()

# Quick timing comparison
print("=== Quick Timing at n=50,000 ===")
print()
big_data = [{"id": i, "value": i * 1.5, "category": "cat_" + str(i % 10)} for i in range(50_000)]
print("Baseline:")
d1 = find_duplicates(big_data)
print("Optimized:")
d2 = find_duplicates_optimized(big_data)
print()
print("The optimized version should be dramatically faster.")

---
## Part 3: Write Tests

In [ ]:
def test_find_duplicates():
    """Test that optimized version matches baseline."""
    # Test 1: No duplicates
    data = [{"id": i, "value": float(i)} for i in range(100)]
    assert len(find_duplicates_optimized(data)) == 0, "No duplicates expected"

    # Test 2: All duplicates
    data = [{"id": i, "value": 42.0} for i in range(100)]
    assert len(find_duplicates_optimized(data)) == 99, "99 duplicates expected"

    # Test 3: Mixed
    data = [{"id": i, "value": float(i % 10)} for i in range(100)]
    assert len(find_duplicates_optimized(data)) == 90, "90 duplicates expected"

    # Test 4: Empty
    assert len(find_duplicates_optimized([])) == 0, "Empty list"

    # Test 5: Single element
    assert len(find_duplicates_optimized([{"value": 1.0}])) == 0, "Single element"

    print("All tests passed!")

test_find_duplicates()

---
## Part 4: Your Turn -- Apply to Your Track

Now apply this same pattern to YOUR project:

1. **Identify** the slowest function in your pipeline
2. **State** its current Big-O and why it is slow
3. **Implement** an optimized version using the right data structure
4. **Verify** that the optimized version gives the same results
5. **Next week**: benchmark at 5 sizes and create comparison plots

### Data Structure Selection Guide

| Problem Pattern | Slow Approach | Fast Data Structure | Improvement |
|----------------|---------------|-------------------|-------------|
| "Is x in my collection?" | list scan O(n) | set/dict O(1) | n/1 |
| "Find all items matching key" | list scan O(n) | hash index O(1) | n/1 |
| "Find item in sorted data" | list scan O(n) | binary search O(log n) | n/log n |
| "Find top-k items" | sort all O(n log n) | heap O(n log k) | log n / log k |
| "Remove duplicates" | nested loops O(n^2) | set O(n) | n/1 |
| "Sort by custom criteria" | bubble sort O(n^2) | Timsort O(n log n) | n / log n |

In [ ]:
# TODO: Replace this template with YOUR project's optimization
#
# Step 1: Describe your bottleneck
# BOTTLENECK: ___
# CURRENT BIG-O: O(___)
# REASON: ___
#
# Step 2: Choose your data structure
# CHOSEN DS: ___
# EXPECTED BIG-O: O(___)
#
# Step 3: Implement
# def baseline_version(data):
#     ...
#
# def optimized_version(data):
#     ...
#
# Step 4: Verify
# result_base = baseline_version(test_data)
# result_opt  = optimized_version(test_data)
# assert result_base == result_opt

print("Template ready -- fill in your project's optimization!")

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)